## GPU Check

In [1]:
from IPython.display import HTML
shell = get_ipython()

def adjust_font_size():
  display(HTML('''
  <style>
    body {
      font-size: 16px;
    }
  </style>
  '''))

if adjust_font_size not in shell.events.callbacks['pre_execute']:
  shell.events.register('pre_execute', adjust_font_size)

In [2]:
!nvidia-smi

/bin/bash: nvidia-smi: command not found


# SimCLR

**Binary classifier (known vs unknown)**  
- Train with labeled data: classes 0~5 -> known, unknown (class 6 (regenerated, and broken)).
- SimCLR
  
**Apply to unlabeled set**
- Predict known/unknown on ~14,000 unlabeled images.
- Assign unknown -> class 6.

**Label propagation**
- For the same fish_id, copy labels across scales (0–5 only).
- Skip if predicted unknown.

**SimCLR training**
- Train SimCLR on all images (labeled + unlabeled).
- Extract features (before projection head).

**Final classification (XGBoost)**  
- Combine SimCLR features + metadata (length, river, river point one-hot).
- Train XGBoost.
- Handle class imbalance with weights/oversampling.

## 1. Filling up Regenerated & Broken (label == 6) cases

In [3]:
import os, re, glob
from pathlib import Path
import pandas as pd

# ------------------------
# 0. Define root path
# ------------------------
ROOT_DIR = Path("data/fish/")
IMAGE_DIRS = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir() and p.name.lower().endswith("images")])
print(f"Found image directories: {[d.name for d in IMAGE_DIRS]}")

# ------------------------
# 1. Regex patterns for files and folders
# ------------------------
fname_pat = re.compile(r"^([A-Za-z]+[0-9]+)_([0-9]+)_([0-9]+)\.(?:png|PNG)$")
folder_pat = re.compile(r"^([A-Za-z]+)([0-9]+)images$", re.IGNORECASE)

records, bad_files = [], []

# ------------------------
# 2. Build master index from image folders
# ------------------------
for d in IMAGE_DIRS:
    folder = d.name
    m_folder = folder_pat.match(folder)
    if m_folder:
        river = m_folder.group(1).upper()   # e.g., CU, DU, TO, PO, TU
        point = int(m_folder.group(2))      # e.g., 1~4
    else:
        river, point = folder.upper(), None

    for path in glob.glob(str(d / "*.png")):
        fn = os.path.basename(path)
        m = fname_pat.match(fn)
        if not m:
            bad_files.append(path)
            continue

        fish_id   = int(m.group(2))
        image_idx = int(m.group(3))

        records.append({
            "path": path,
            "file": fn,
            "folder": folder,
            "river": river,
            "point": point,
            "fish_id": fish_id,
            "image_idx": image_idx
        })

df_master = pd.DataFrame(records)
print(f"\n[INFO] Total PNG parsed: {len(df_master)}")
if bad_files:
    print(f"[WARN] Skipped (name pattern mismatch): {len(bad_files)}")

print("\n[Preview] Master index (before merge):")
print(df_master.head())

# ------------------------
# 3. Merge label.csv (match by file name)
# ------------------------
label = pd.read_csv("data/fish/label.csv")
print("\n[INFO] label.csv preview:")
print(label.head())

# Create file_stem for matching
df_master["file_stem"] = df_master["file"].apply(lambda fn: Path(fn).stem)
df_master = df_master.merge(label, left_on="file_stem", right_on="name", how="left")
df_master.drop(columns=["file_stem","name"], inplace=True)

# ------------------------
# 4. Merge trout_demo.csv (length info)
# ------------------------
demo = pd.read_csv("data/fish/trout_demo.csv")
print("\n[INFO] trout_demo.csv preview:")
print(demo.head())

# Create merge key: river(lowercase) + point + "_" + fish_id
df_master["demo_key"] = (
    df_master["river"].str.lower()
    + df_master["point"].astype(str)
    + "_" + df_master["fish_id"].astype(str)
)

df_master = df_master.merge(
    demo[["river","length"]], left_on="demo_key", right_on="river", how="left"
)

# Clean up columns
df_master.drop(columns=["demo_key","river_y"], inplace=True)
df_master.rename(columns={"river_x":"river"}, inplace=True)

print("\n[Preview] Master index (after merge):")
print(df_master.head())
print(df_master.shape)

# ------------------------
# 5. Save merged dataset
# ------------------------
# out_path = ROOT_DIR / "simCLR_endtoend/dataset_master.csv"
# df_master.to_csv(out_path, index=False)
# print(f"\n[INFO] Saved merged master dataset: {out_path}")


Found image directories: ['cu1images', 'cu2images', 'cu3images', 'cu4images', 'du1images', 'du2images', 'du3images', 'du4images', 'po1images', 'po2images', 'po3images', 'po4images', 'to1images', 'to2images', 'to3images', 'to4images', 'tu1images', 'tu2images', 'tu3images', 'tu4images']

[INFO] Total PNG parsed: 16221

[Preview] Master index (before merge):
                                path           file     folder river  point  \
0   data/fish/cu1images/cu1_51_7.png   cu1_51_7.png  cu1images    CU      1   
1  data/fish/cu1images/cu1_30_06.png  cu1_30_06.png  cu1images    CU      1   
2  data/fish/cu1images/cu1_65_08.png  cu1_65_08.png  cu1images    CU      1   
3  data/fish/cu1images/cu1_63_06.png  cu1_63_06.png  cu1images    CU      1   
4  data/fish/cu1images/cu1_79_01.png  cu1_79_01.png  cu1images    CU      1   

   fish_id  image_idx  
0       51          7  
1       30          6  
2       65          8  
3       63          6  
4       79          1  

[INFO] label.csv previ

In [4]:
import numpy as np

# label.isna() == 0, label.notna == 1
df_master['split'] = np.where(df_master['label'].notna(), 1, 0)

print(df_master.shape)
df_master.head()

(16221, 10)


,path,file,folder,river,point,fish_id,image_idx,label,length,split
0,data/fish/cu1images/cu1_51_7.png,cu1_51_7.png,cu1images,CU,1,51,7,NaN,178,0
1,data/fish/cu1images/cu1_30_06.png,cu1_30_06.png,cu1images,CU,1,30,6,NaN,116,0
2,data/fish/cu1images/cu1_65_08.png,cu1_65_08.png,cu1images,CU,1,65,8,NaN,169,0
3,data/fish/cu1images/cu1_63_06.png,cu1_63_06.png,cu1images,CU,1,63,6,NaN,228,0
4,data/fish/cu1images/cu1_79_01.png,cu1_79_01.png,cu1images,CU,1,79,1,NaN,234,0


In [5]:
df_master[(df_master['river'] == 'CU') & (df_master['point'] == 1) & (df_master['fish_id'] == 5) & (df_master['label'].notna())]

,path,file,folder,river,point,fish_id,image_idx,label,length,split
251,data/fish/cu1images/cu1_5_03.png,cu1_5_03.png,cu1images,CU,1,5,3,1.0,209,1
407,data/fish/cu1images/cu1_5_05.png,cu1_5_05.png,cu1images,CU,1,5,5,1.0,209,1
818,data/fish/cu1images/cu1_5_04.png,cu1_5_04.png,cu1images,CU,1,5,4,1.0,209,1
868,data/fish/cu1images/cu1_5_07.png,cu1_5_07.png,cu1images,CU,1,5,7,6.0,209,1


In [6]:
df_master[(df_master['river'] == 'CU') & (df_master['point'] == 1) & (df_master['fish_id'] == 5) & (df_master['label'].isna())]

,path,file,folder,river,point,fish_id,image_idx,label,length,split
16,data/fish/cu1images/cu1_5_06.png,cu1_5_06.png,cu1images,CU,1,5,6,NaN,209,0
988,data/fish/cu1images/cu1_5_02.png,cu1_5_02.png,cu1images,CU,1,5,2,NaN,209,0
992,data/fish/cu1images/cu1_5_01.png,cu1_5_01.png,cu1images,CU,1,5,1,NaN,209,0


**Splite Full-Dataset into Labeled and Unlabeled Dataset**

Because we are not able to predict regenerated cases or broken cases and age at the same time. So, we need to deal with regenerated and broken cases first.

In [7]:
labeled = df_master[df_master['split'] == 1].reset_index(drop = True)
unlabeled = df_master[df_master['split'] == 0].reset_index(drop=True)

print('labeled dataset:', labeled.shape)
print('unlabeled dataset:', unlabeled.shape)

labeled dataset: (1393, 10)
unlabeled dataset: (14828, 10)


In [8]:
labeled.head()

,path,file,folder,river,point,fish_id,image_idx,label,length,split
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,1
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,1
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,1


In [9]:
labeled.label.value_counts().sort_index()

label
0.0    405
1.0    533
2.0    185
3.0    108
4.0     42
5.0     13
6.0    107
Name: count, dtype: int64

In [10]:
labeled.shape

(1393, 10)

In [11]:
# label == 6 (regenerated & broken) => 1
# label == 0~5 => 0
labeled['known'] = np.where(labeled['label'] == 6, 1, 0)

### Data Split

split into `Regenerated or Broken` and `Good` 

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    labeled,
    test_size = 0.2,               # split 8:2
    stratify = labeled['known'],
    random_state = 100
)

train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain distribution:\n", train['known'].value_counts())
print("\nTest distribution:\n", test['known'].value_counts())

Train shape: (1114, 11)
Test shape : (279, 11)

Train distribution:
 known
0    1028
1      86
Name: count, dtype: int64

Test distribution:
 known
0    258
1     21
Name: count, dtype: int64


**Preprocessing**

In [13]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),           # Crop
    transforms.RandomHorizontalFlip(),           # Flip
    transforms.ColorJitter(0.8, 0.8, 0.8, 0.2),  # Color
    transforms.RandomGrayscale(p = 0.2),         # Gray scale
    transforms.ToTensor(),                       
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])  
                    # Normalize: mean and std are from ResNet18 model.
])

In [14]:
# https://dilithjay.com/blog/nt-xent-loss-explained
import torch.nn as nn

class NTXentLoss(nn.Module):
    def __init__(self, batch_size, temperature=0.5, device='cuda'):
        super(NTXentLoss, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

        self.mask = self._get_correlated_mask().to(device)

    def _get_correlated_mask(self):
        N = 2 * self.batch_size
        mask = torch.ones((N, N), dtype=bool)
        mask = mask.fill_diagonal_(0)
        for i in range(self.batch_size):
            mask[i, self.batch_size + i] = 0
            mask[self.batch_size + i, i] = 0
        return mask

    def forward(self, z1, z2):
        N = 2 * self.batch_size
        z = torch.cat((z1, z2), dim=0)

        # Cosine similarity
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

        # Positive pair
        sim_ij = torch.diag(sim, self.batch_size)
        sim_ji = torch.diag(sim, -self.batch_size)
        positives = torch.cat([sim_ij, sim_ji], dim=0)

        # Negative pair
        negatives = sim[self.mask].view(N, -1)

        # Logits and labels
        logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
        labels = torch.zeros(N, dtype=torch.long).to(self.device)

        loss = self.criterion(logits / self.temperature, labels)
        return loss / N


In [15]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SimCLRDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert("RGB")

        xi = self.transform(img)
        xj = self.transform(img)

        return xi, xj

train_transform = SimCLRDataset(train, transform = simclr_transform)

**simCLR Pretraining**

In [16]:
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader  = DataLoader(train_transform,
                           batch_size  = 128,
                           shuffle     = True,
                           num_workers = 12,     # the number of CPU
                           drop_last   = True)

backbone = models.resnet18(pretrained = True)  # pretrained = True
backbone.fc = nn.Identity()  # (batch, 512) feature
backbone= backbone.to(device)

projection_head = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 128)
).to(device)

# 3. Contrastive loss + optimizer
optimizer = torch.optim.Adam(list(backbone.parameters()) + list(projection_head.parameters()), lr = 3e-4) # Learning Rate

criterion = NTXentLoss(batch_size = 128,
                       temperature = 0.2)

from tqdm import tqdm

for epoch in range(50):                     # Epoch 100
    loop = tqdm(train_loader, leave = True)
    for x1, x2 in loop:
        x1, x2 = x1.to(device), x2.to(device)

        # 1) extract backbone feature
        h1 = backbone(x1)   # flatten fe ature (128d)
        h2 = backbone(x2)

        # 2) projection head (contrastive for training)
        z1 = projection_head(h1)       # (batch, 128)
        z2 = projection_head(h2)

        # 3) compute contrastive loss
        loss = criterion(z1, z2)

        # 4) backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_description(f'Epoch [{epoch+1}/50]')
        loop.set_postfix(loss=loss.item())

/home/jlc3q/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jlc3q/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
  0%|          | 0/8 [00:02<?, ?it/s]


KeyboardInterrupt: 

**Save Backbone Result**

Save total simCLR structure and backbone

In [17]:
# save_path = "data/fish/simCLR_endtoend/backbone_resnet18_simclr.pth"
# torch.save(backbone, save_path)
# print(f"Backbone saved at {save_path}")

Backbone saved at data/fish/simCLR_endtoend/backbone_resnet18_simclr.pth


**Using Pre-trained Model**

In [16]:
import torch
import torchvision.models as models
import torch.nn as nn

# load saved backbone
save_path = "data/fish/simCLR_endtoend/backbone_resnet18_simclr.pth"
backbone = torch.load(save_path, map_location='cuda', weights_only=False)
backbone.eval()

print("✅ Backbone successfully loaded from:", save_path)

✅ Backbone successfully loaded from: data/fish/simCLR_endtoend/backbone_resnet18_simclr.pth


expanding the feature extraction stage for transition to fine-tuning (end-to-end supervised learning)

In [17]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = nn.Sequential(
    backbone,                      
    nn.Flatten()                   # (B, 512)
).to(device)  

Adding label

In [18]:
import torch, random, numpy as np

def set_seed(seed = 100):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# holding seed
set_seed(100)


num_classes = len(train['known'].unique())  # train['known'] is an label

classifier_head = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
).to(device)

model = nn.Sequential(backbone, classifier_head)

The current structure is:  
**Input Images -> SimCLR backbone -> `Flatten` -> Classifier -> Prediction**

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Fine-tuning (Dataset & Dataloader)

In [20]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class LabeledDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        x = self.transform(img)
        y = int(row['known'])
        return x, y

set_seed(100) 

train_dataset = LabeledDataset(train, transform)
test_dataset  = LabeledDataset(test, transform)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, num_workers = 10)
test_loader  = DataLoader(test_dataset, batch_size = 64, shuffle = False, num_workers = 10)


Learning Loop

In [21]:
from tqdm import tqdm

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)

        preds = model(imgs)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")


Epoch [1/20]: 100%|██████████| 18/18 [00:05<00:00,  3.17it/s, loss=0.138]


Epoch 1 | Avg Loss: 0.3034


Epoch [2/20]: 100%|██████████| 18/18 [00:05<00:00,  3.25it/s, loss=0.21]  


Epoch 2 | Avg Loss: 0.1235


Epoch [3/20]: 100%|██████████| 18/18 [00:05<00:00,  3.24it/s, loss=0.0425]


...

Epoch 18 | Avg Loss: 0.0003


Epoch [19/20]: 100%|██████████| 18/18 [00:05<00:00,  3.35it/s, loss=0.000263]


Epoch 19 | Avg Loss: 0.0003


Epoch [20/20]: 100%|██████████| 18/18 [00:05<00:00,  3.21it/s, loss=0.000161]

Epoch 20 | Avg Loss: 0.0003


Evaluation

In [22]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       258
           1       0.80      0.57      0.67        21

    accuracy                           0.96       279
   macro avg       0.88      0.78      0.82       279
weighted avg       0.95      0.96      0.95       279



In [23]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
import numpy as np
import pandas as pd

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class UnlabeledDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        img = Image.open(img_path).convert("RGB")
        x = self.transform(img)
        return x

# DataLoader
unlabeled_dataset = UnlabeledDataset(unlabeled, transform)
unlabeled_loader  = DataLoader(unlabeled_dataset, 
                               batch_size = 64, 
                               shuffle = False, 
                               num_workers = 10)

# Prediction
model.eval()
preds_all = []

with torch.no_grad():
    for imgs in unlabeled_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)      
        preds_all.extend(preds.cpu().numpy())

unlabeled['predicted_label'] = preds_all
unlabeled['predicted_label'].value_counts()

predicted_label
0    10982
1     3846
Name: count, dtype: int64

In [24]:
unlabeled.head()

,path,file,folder,river,point,fish_id,image_idx,label,length,split,predicted_label
0,data/fish/cu1images/cu1_51_7.png,cu1_51_7.png,cu1images,CU,1,51,7,NaN,178,0,1
1,data/fish/cu1images/cu1_30_06.png,cu1_30_06.png,cu1images,CU,1,30,6,NaN,116,0,1
2,data/fish/cu1images/cu1_65_08.png,cu1_65_08.png,cu1images,CU,1,65,8,NaN,169,0,0
3,data/fish/cu1images/cu1_63_06.png,cu1_63_06.png,cu1images,CU,1,63,6,NaN,228,0,1
4,data/fish/cu1images/cu1_79_01.png,cu1_79_01.png,cu1images,CU,1,79,1,NaN,234,0,0


In [25]:
display(unlabeled[['path', 'predicted_label']].head())
print(unlabeled['predicted_label'].value_counts())

,path,predicted_label
0,data/fish/cu1images/cu1_51_7.png,1
1,data/fish/cu1images/cu1_30_06.png,1
2,data/fish/cu1images/cu1_65_08.png,0
3,data/fish/cu1images/cu1_63_06.png,1
4,data/fish/cu1images/cu1_79_01.png,0


predicted_label
0    10982
1     3846
Name: count, dtype: int64


- Uisng pretrained backbon (Unlabeled Images)

In [26]:
print('The # of Original Label 6: {}'.format(labeled[labeled['label'] == 6].shape[0]))

The # of Original Label 6: 107


In [27]:
print("The number of original labeled images is 107, and now we have additional labeled images ({}) from unlabeled images".\
                          format(unlabeled['predicted_label'].value_counts()[1] - 
                                 labeled[labeled['label'] == 6].shape[0]))

The number of original labeled images is 107, and now we have additional labeled images (3739) from unlabeled images


In [28]:
labeled['source'] = 'labeled'
labeled = labeled.copy()

unlabeled['source'] = 'unlabeled'

display(labeled.head(2))
display(unlabeled.head(2))

,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,0,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,0,labeled


,path,file,folder,river,point,fish_id,image_idx,label,length,split,predicted_label,source
0,data/fish/cu1images/cu1_51_7.png,cu1_51_7.png,cu1images,CU,1,51,7,NaN,178,0,1,unlabeled
1,data/fish/cu1images/cu1_30_06.png,cu1_30_06.png,cu1images,CU,1,30,6,NaN,116,0,1,unlabeled


In [29]:
unlabeled2 = unlabeled.copy()
unlabeled2['label'] = unlabeled2['predicted_label'].map({1: 6, 0: np.nan})

unlabeled2 = unlabeled2.drop(columns=['predicted_label'])

df_all_combined = pd.concat([labeled, unlabeled2],
                            axis = 0,
                            ignore_index = True)

print('Combined shape:', df_all_combined.shape)

display(df_all_combined.head(5))

Combined shape: (16221, 12)


,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,0.0,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,0.0,labeled
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,1,1.0,labeled
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,1,0.0,labeled
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,1,0.0,labeled


In [30]:
print(df_all_combined.source.value_counts())

source
unlabeled    14828
labeled       1393
Name: count, dtype: int64


In [31]:
df_all_combined[(df_all_combined['source'] == 'unlabeled') & (df_all_combined['label'] == 6)].shape

(3846, 12)

**Save Prediction Results (Prediction of Broken Label to Unlabeled Images)**

In [32]:
# df_all_combined.to_csv('data/fish/simCLR_endtoend/pre_all.csv', index = False)

## 2. Extending Label (Based on Same River, Point, and Fish_id)

- Bringing in labels from experts.
- Assigning labels to the same group based on river, point, and trout number.

**Load Combined Dataset Created Above**

In [4]:
import pandas as pd
combined_df = pd.read_csv('data/fish/simCLR_endtoend/pre_all.csv')

print(combined_df.shape)
combined_df.head()

(16221, 12)


,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,0.0,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,0.0,labeled
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,1,1.0,labeled
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,1,0.0,labeled
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,1,0.0,labeled


In [5]:
print(combined_df[combined_df['label'].isna()].shape)
combined_df[combined_df['label'].isna()].head()

(10982, 12)


,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source
1395,data/fish/cu1images/cu1_65_08.png,cu1_65_08.png,cu1images,CU,1,65,8,NaN,169,0,NaN,unlabeled
1397,data/fish/cu1images/cu1_79_01.png,cu1_79_01.png,cu1images,CU,1,79,1,NaN,234,0,NaN,unlabeled
1399,data/fish/cu1images/cu1_38_03.png,cu1_38_03.png,cu1images,CU,1,38,3,NaN,114,0,NaN,unlabeled
1402,data/fish/cu1images/cu1_23_02.png,cu1_23_02.png,cu1images,CU,1,23,2,NaN,121,0,NaN,unlabeled
1403,data/fish/cu1images/cu1_97_03.png,cu1_97_03.png,cu1images,CU,1,97,3,NaN,231,0,NaN,unlabeled


**Extending Rules (Summary)**

**Target:** `combined_df`

**Label Filling Conditions**
- Only fill when `source == 'unlabeled'` **AND** `label.isna()`.
- If `label == 10`, always exclude (do not use as reference and do not fill).
- If `source == 'unlabeled'` but `label` already exists, leave it unchanged.

**Filling Criteria**
- Key for matching: `(river, point, fish_id)`
- Use `source == 'labeled'` data for reference.
- Ignore any `label == 10` entries when calculating the majority.

**Conflict Handling**
- If a majority label exists → fill with that label.
- If there is a tie (e.g., 1:1) → do not fill and save these cases in `conflict_cases`.


In [6]:
import pandas as pd

# Copy the original DataFrame to avoid modifying it directly
combined_df2 = combined_df.copy()

# Select only labeled data (exclude label == 10 because it's not usable for propagation)
labeled_df = combined_df2[
    (combined_df2["source"] == "labeled") & (combined_df2["label"] != 6)
]

# Identify unlabeled rows where label is missing (NaN)
target_mask = (combined_df2["source"] == "unlabeled") & (combined_df2["label"].isna())

# List to collect conflict cases (tie situations)
conflict_list = []

# Group labeled data by (river, point, fish_id)
grouped = labeled_df.groupby(["river", "point", "fish_id"])["label"]

# Count label occurrences within each group
label_counts = grouped.value_counts().unstack(fill_value = 0)

# Iterate over each group
for key, row in label_counts.iterrows():
    river, point, fish_id = key
    counts = row[row > 0]  # consider only labels with count > 0

    if len(counts) == 0:
        continue  # no labels available → skip

    # Majority check
    max_count = counts.max()
    top_labels = counts[counts == max_count].index.tolist()

    if len(top_labels) == 1:  # majority exists
        fill_label = top_labels[0]
        mask = (
            (combined_df2["river"] == river) &
            (combined_df2["point"] == point) &
            (combined_df2["fish_id"] == fish_id) &
            target_mask
        )
        combined_df2.loc[mask, "label"] = fill_label
    else:  # tie situation → save to conflict_cases
        conflict_list.append({
            "river": river,
            "point": point,
            "fish_id": fish_id,
            "candidate_labels": top_labels
        })

# Create a DataFrame for conflict cases
conflict_cases = pd.DataFrame(conflict_list)

**Conflict Cases**

In [7]:
conflict_cases.head()

,river,point,fish_id,candidate_labels
0,DU,3,11,"[0.0, 1.0]"
1,DU,3,31,"[0.0, 1.0]"
2,PO,2,77,"[0.0, 1.0]"
3,TO,4,13,"[3.0, 4.0]"
4,TU,4,46,"[0.0, 1.0]"


In [8]:
# Count only originally labeled rows
original_labeled_count = ((combined_df["label"].notna()) &
                          (combined_df["source"] == "labeled")).sum()

# Count unlabeled rows that already had labels before filling
original_unlabeled_count = ((combined_df["label"].notna()) &
                            (combined_df["source"] == "unlabeled")).sum()

# Count newly filled labels (excluding label == 6)
newly_filled_count = ((combined_df2["label"].notna()) &
                      (combined_df2["source"] == "unlabeled") &
                      (combined_df2["label"] != 6)).sum()

# Final total number of labels after filling
final_total_count = combined_df2["label"].notna().sum()

print("✅ Label filling completed")
print("Original labeled count        :", original_labeled_count)
print("Add Label == 6                :", original_unlabeled_count)
print("Extend Label count            :", newly_filled_count)
print("Final total labeled count     :", final_total_count)
print("Number of conflict cases      :", len(conflict_cases))

✅ Label filling completed
Original labeled count        : 1393
Add Label == 6                : 3846
Extend Label count            : 4743
Final total labeled count     : 9982
Number of conflict cases      : 5


**Check Label Distribution**

In [9]:
combined_df2.groupby('source')['label'].value_counts()

source     label
labeled    1.0       533
           0.0       405
           2.0       185
           3.0       108
           6.0       107
           4.0        42
           5.0        13
unlabeled  6.0      3846
           1.0      2391
           0.0      1474
           2.0       578
           3.0       192
           4.0        81
           5.0        27
Name: count, dtype: int64

In [10]:
print(combined_df2.shape)
combined_df2.head()

(16221, 12)


,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,0.0,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,0.0,labeled
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,1,1.0,labeled
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,1,0.0,labeled
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,1,0.0,labeled


## 3. Training SimCLR with New Labeled Dataset (Labeled + New Broken Labeled)

In [11]:
labeled_combined = combined_df2[combined_df2['label'].notna()].reset_index(drop = True)
unlabeled_combined = combined_df2[combined_df2['label'].isna()].reset_index(drop = True)

print('labeled dataset:', labeled_combined.shape)
print('unlabeled dataset:', unlabeled_combined.shape)

labeled dataset: (9982, 12)
unlabeled dataset: (6239, 12)


**Data Split**

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Make a group identifier
labeled_combined['group_id'] = (
    labeled_combined['river'].astype(str) + '_' +
    labeled_combined['point'].astype(str) + '_' +
    labeled_combined['fish_id'].astype(str)
)

X = labeled_combined.drop(columns = ['label'])
y = labeled_combined['label']

groups = labeled_combined['group_id']

# Give Group Effect
gss = GroupShuffleSplit(n_splits = 1,
                        test_size = 0.2,      # 8:2
                        random_state = 100)

for train_idx, test_idx in gss.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain distribution:\n", y_train.value_counts().sort_index())
print("\nTest distribution:\n", y_test.value_counts().sort_index())

Train shape: (7864, 12)
Test shape : (2118, 12)

Train distribution:
 label
0.0    1509
1.0    2201
2.0     628
3.0     231
4.0     115
5.0      30
6.0    3150
Name: count, dtype: int64

Test distribution:
 label
0.0    370
1.0    723
2.0    135
3.0     69
4.0      8
5.0     10
6.0    803
Name: count, dtype: int64


In [13]:
X_test.shape

(2118, 12)

In [14]:
train_only_df = combined_df2.drop(index=test_idx)
train_only_df.shape

(14103, 12)

In [76]:
# train_only_df = combined_df2.drop(index=test_idx)

# combined_df2['streamlit'] = 0
# combined_df2.loc[train_only_df.index, 'streamlit'] = 1

In [77]:
# combined_df2.to_csv('data/fish/simCLR_endtoend/streamlib.csv', index = False)

**SimCLR Preprocessing**

In [15]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.8, 0.8, 0.8, 0.2),
    transforms.RandomGrayscale(p = 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225])
])

In [16]:
# https://dilithjay.com/blog/nt-xent-loss-explained
import torch.nn as nn

class NTXentLoss(nn.Module):
    def __init__(self, batch_size, temperature=0.5, device='cuda'):
        super(NTXentLoss, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

        self.mask = self._get_correlated_mask().to(device)

    def _get_correlated_mask(self):
        N = 2 * self.batch_size
        mask = torch.ones((N, N), dtype=bool)
        mask = mask.fill_diagonal_(0)
        for i in range(self.batch_size):
            mask[i, self.batch_size + i] = 0
            mask[self.batch_size + i, i] = 0
        return mask

    def forward(self, z1, z2):
        N = 2 * self.batch_size
        z = torch.cat((z1, z2), dim=0)

        # Cosine similarity
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

        # Positive pair
        sim_ij = torch.diag(sim, self.batch_size)
        sim_ji = torch.diag(sim, -self.batch_size)
        positives = torch.cat([sim_ij, sim_ji], dim=0)

        # Negative pair
        negatives = sim[self.mask].view(N, -1)

        # Logits and labels
        logits = torch.cat([positives.unsqueeze(1), negatives], dim=1)
        labels = torch.zeros(N, dtype=torch.long).to(self.device)

        loss = self.criterion(logits / self.temperature, labels)
        return loss / N


In [17]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SimCLRDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert("RGB")

        xi = self.transform(img)
        xj = self.transform(img)

        return xi, xj

train_transform = SimCLRDataset(X_train, transform = simclr_transform)

**SimCLR Pre-Training**

In [47]:
import torchvision.models as models
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader  = DataLoader(train_transform,
                           batch_size  = 128,
                           shuffle     = True,
                           num_workers = 10,
                           pin_memory=True,
                           drop_last   = True)

#  SimCLR model
# backbone = CNN(feature_dim = 128).to(device) -> too simple model

backbone = models.resnet18(pretrained = True)  # pretrained=True
backbone.fc = nn.Identity()  # (batch, 512) feature
backbone= backbone.to(device)

projection_head = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, 128)
).to(device)

# 3. Contrastive loss + optimizer
optimizer = torch.optim.Adam(list(backbone.parameters()) +list(projection_head.parameters()), lr = 3e-4)

criterion = NTXentLoss(batch_size = 128,
                       temperature = 0.2)

from tqdm import tqdm

for epoch in range(30):
    loop = tqdm(train_loader, leave=True)
    for x1, x2 in loop:
        x1, x2 = x1.to(device), x2.to(device)

        # 1) extract backbone feature
        h1 = backbone(x1)   # flatten fe ature (128d)
        h2 = backbone(x2)

        # 2) projection head (contrastive for training)
        z1 = projection_head(h1)       # (batch, 128)
        z2 = projection_head(h2)

        # 3) compute contrastive loss
        loss = criterion(z1, z2)

        # 4) backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_description(f'Epoch [{epoch+1}/30]')
        loop.set_postfix(loss=loss.item())

/home/jlc3q/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jlc3q/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch [30/30]: 100%|██████████| 61/61 [00:41<00:00,  1.47it/s, loss=2.25]


**Save new backbon**

In [48]:
# save_path = "data/fish/simCLR_endtoend/backbone_resnet18_simclr2.pth"
# torch.save(backbone, save_path)
# print(f"Backbone saved at {save_path}")

Backbone saved at data/fish/simCLR_endtoend/backbone_resnet18_simclr2.pth


**Load New Backbon**

In [18]:
import torch
import torchvision.models as models
import torch.nn as nn

# load saved backbone
save_path = "data/fish/simCLR_endtoend/backbone_resnet18_simclr2.pth"
backbone = torch.load(save_path, map_location='cuda', weights_only=False)
backbone.eval()

print("✅ Backbone successfully loaded from:", save_path)

✅ Backbone successfully loaded from: data/fish/simCLR_endtoend/backbone_resnet18_simclr2.pth


In [19]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = nn.Sequential(
    backbone,                      
    nn.Flatten()                   # (B, 512)
).to(device)  

In [20]:
num_classes = len(y_train.unique())  # X_test is an label

classifier_head = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
).to(device)

model = nn.Sequential(backbone, classifier_head)

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [22]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class LabeledDatasetXY(Dataset):
    def __init__(self, X, y, transform):
        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        
        if isinstance(self.X, pd.DataFrame):
            img_path = self.X.iloc[idx]['path']  # DataFrame
        else:
            img_path = self.X.iloc[idx]          # Series

        img = Image.open(img_path).convert("RGB")
        x = self.transform(img)
        y = int(self.y.iloc[idx])
        return x, y
        
train_dataset = LabeledDatasetXY(X_train, y_train, transform)
test_dataset  = LabeledDatasetXY(X_test, y_test, transform)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, num_workers = 10)
test_loader  = DataLoader(test_dataset, batch_size = 64, shuffle = False, num_workers = 10)

In [68]:
from tqdm import tqdm

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)

        # Forward pass
        preds = model(imgs)
        loss = criterion(preds, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")


Epoch [1/10]: 100%|██████████| 123/123 [00:35<00:00,  3.43it/s, loss=0.419]


Epoch 1 | Avg Loss: 0.6670


Epoch [2/10]: 100%|██████████| 123/123 [00:35<00:00,  3.47it/s, loss=0.357]


Epoch 2 | Avg Loss: 0.3380


Epoch [3/10]: 100%|██████████| 123/123 [00:35<00:00,  3.50it/s, loss=0.164]


...

Epoch 8 | Avg Loss: 0.0196


Epoch [9/10]: 100%|██████████| 123/123 [00:35<00:00,  3.48it/s, loss=0.00621]


Epoch 9 | Avg Loss: 0.0151


Epoch [10/10]: 100%|██████████| 123/123 [00:35<00:00,  3.44it/s, loss=0.00896]

Epoch 10 | Avg Loss: 0.0162


Save Fine-tuning values

In [69]:
# Fine-tuning 
# torch.save(classifier_head.state_dict(), "data/fish/simCLR_endtoend/classifier_head.pth")

# torch.save({
#     "backbone_state_dict": backbone.state_dict(),
#     "head_state_dict": classifier_head.state_dict()
# }, "data/fish/simCLR_endtoend/backbone_head.pth")

In [33]:
checkpoint = torch.load("data/fish/simCLR_endtoend/backbone_head.pth", map_location=device)
# checkpoint = torch.load("data/fish/simCLR_endtoend/simCLR_endtoend-backbone_head_v1.pth", map_location=device)
# checkpoint = torch.load("data/fish/simCLR_endtoend/simCLR_endtoend-backbone_head_v2.pth", map_location=device)
# checkpoint = torch.load("data/fish/simCLR_endtoend/simCLR_endtoend-backbone_head_v5.pth", map_location=device)

backbone.load_state_dict(checkpoint["backbone_state_dict"])
classifier_head.load_state_dict(checkpoint["head_state_dict"])

<All keys matched successfully>

In [34]:
model = nn.Sequential(backbone, classifier_head).to(device)

In [30]:
# original
from sklearn.metrics import classification_report
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86       370
           1       0.84      0.81      0.82       723
           2       0.48      0.36      0.41       135
           3       0.47      0.46      0.47        69
           4       1.00      0.25      0.40         8
           5       0.38      0.30      0.33        10
           6       0.95      0.97      0.96       803

    accuracy                           0.84      2118
   macro avg       0.70      0.58      0.61      2118
weighted avg       0.84      0.84      0.84      2118



In [27]:
# v1
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.90      0.87       370
           1       0.86      0.80      0.83       723
           2       0.47      0.40      0.43       135
           3       0.49      0.57      0.52        69
           4       1.00      0.25      0.40         8
           5       0.29      0.20      0.24        10
           6       0.93      0.98      0.96       803

    accuracy                           0.85      2118
   macro avg       0.70      0.58      0.61      2118
weighted avg       0.84      0.85      0.84      2118



In [35]:
# v5
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.22      0.35       370
           1       0.97      0.21      0.34       723
           2       0.53      0.34      0.42       135
           3       0.30      0.55      0.39        69
           4       0.00      0.00      0.00         8
           5       0.00      0.00      0.00        10
           6       0.48      1.00      0.65       803

    accuracy                           0.53      2118
   macro avg       0.45      0.33      0.31      2118
weighted avg       0.71      0.53      0.46      2118



/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [67]:
# v3
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.82      0.84       370
           1       0.96      0.32      0.47       723
           2       0.24      0.94      0.38       135
           3       0.88      0.10      0.18        69
           4       0.00      0.00      0.00         8
           5       0.33      0.20      0.25        10
           6       0.80      0.99      0.89       803

    accuracy                           0.69      2118
   macro avg       0.58      0.48      0.43      2118
weighted avg       0.83      0.69      0.68      2118



/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Evaluation

In [70]:
from sklearn.metrics import classification_report

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86       370
           1       0.84      0.81      0.82       723
           2       0.48      0.36      0.41       135
           3       0.47      0.46      0.47        69
           4       1.00      0.25      0.40         8
           5       0.38      0.30      0.33        10
           6       0.95      0.97      0.96       803

    accuracy                           0.84      2118
   macro avg       0.70      0.58      0.61      2118
weighted avg       0.84      0.84      0.84      2118



**Final Prediction (Label 0~5)**

In [58]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
import numpy as np
import pandas as pd

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class UnlabeledDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop = True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        img = Image.open(img_path).convert("RGB")
        x = self.transform(img)
        return x

# DataLoader
unlabeled_dataset2 = UnlabeledDataset(unlabeled_combined, transform)
unlabeled_loader2  = DataLoader(unlabeled_dataset2, 
                                batch_size = 64, 
                                shuffle = False, 
                                num_workers = 10)

# Prediction
model.eval()
preds_all = []

with torch.no_grad():
    for imgs in unlabeled_loader2:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)      
        preds_all.extend(preds.cpu().numpy())

unlabeled_combined['predicted_label'] = preds_all
unlabeled_combined['predicted_label'].value_counts()

predicted_label
1    4312
0     998
2     411
6     407
3      54
5      29
4      28
Name: count, dtype: int64

**Combine Labeled Images and New labeled Images**

In [59]:
unlabeled_combined.head()

,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source,predicted_label
0,data/fish/cu1images/cu1_65_08.png,cu1_65_08.png,cu1images,CU,1,65,8,NaN,169,0,NaN,unlabeled,1
1,data/fish/cu1images/cu1_79_01.png,cu1_79_01.png,cu1images,CU,1,79,1,NaN,234,0,NaN,unlabeled,1
2,data/fish/cu1images/cu1_38_03.png,cu1_38_03.png,cu1images,CU,1,38,3,NaN,114,0,NaN,unlabeled,1
3,data/fish/cu1images/cu1_97_03.png,cu1_97_03.png,cu1images,CU,1,97,3,NaN,231,0,NaN,unlabeled,1
4,data/fish/cu1images/cu1_112_11.png,cu1_112_11.png,cu1images,CU,1,112,11,NaN,120,0,NaN,unlabeled,1


In [60]:
labeled_combined.head(2)

,path,file,folder,river,point,fish_id,image_idx,label,length,split,known,source,group_id
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,0.0,labeled,CU_1_22
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,0.0,labeled,CU_1_12


In [61]:
unlabeled_combined2 = unlabeled_combined.copy()

In [62]:
# 1) Process unlabeled_combined2
unlabeled_combined2_processed = unlabeled_combined2.copy()

# Fill NaN label with predicted_label
unlabeled_combined2_processed["label"] = unlabeled_combined2_processed["label"].fillna(
    unlabeled_combined2_processed["predicted_label"]
)

# Drop helper columns
unlabeled_combined2_processed = unlabeled_combined2_processed.drop(
    columns=["predicted_label"]
)

# 2) Process labeled_combined
labeled_combined_processed = labeled_combined.copy()
if "group_id" in labeled_combined_processed.columns:
    labeled_combined_processed = labeled_combined_processed.drop(columns=["group_id"])

# 3) Keep only desired columns
cols_to_keep = ["path", "file", "folder", "river", "point",
                "fish_id", "image_idx", "label", "length", "split", "source"]

unlabeled_combined2_processed = unlabeled_combined2_processed[cols_to_keep]
labeled_combined_processed    = labeled_combined_processed[cols_to_keep]

# 4) Concatenate
combined_final = pd.concat(
    [labeled_combined_processed, unlabeled_combined2_processed],
    axis=0
).reset_index(drop=True)

print("Final combined shape:", combined_final.shape)
combined_final.head()


Final combined shape: (16221, 11)


,path,file,folder,river,point,fish_id,image_idx,label,length,split,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,1,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,1,labeled
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,1,labeled
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,1,labeled
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,1,labeled


In [63]:
del combined_final['split']

In [64]:
combined_final.head()

,path,file,folder,river,point,fish_id,image_idx,label,length,source
0,data/fish/cu1images/cu1_22_02.png,cu1_22_02.png,cu1images,CU,1,22,2,4.0,243,labeled
1,data/fish/cu1images/cu1_12_07.png,cu1_12_07.png,cu1images,CU,1,12,7,2.0,243,labeled
2,data/fish/cu1images/cu1_109_04.png,cu1_109_04.png,cu1images,CU,1,109,4,6.0,295,labeled
3,data/fish/cu1images/cu1_37_02.png,cu1_37_02.png,cu1images,CU,1,37,2,1.0,124,labeled
4,data/fish/cu1images/cu1_116_05.png,cu1_116_05.png,cu1images,CU,1,116,5,0.0,113,labeled


In [65]:
combined_final[(combined_final['source'] == 'unlabeled') & (combined_final['label'].notna())].tail()

,path,file,folder,river,point,fish_id,image_idx,label,length,source
16216,data/fish/tu4images/tu4_61_02.png,tu4_61_02.png,tu4images,TU,4,61,2,1.0,143,unlabeled
16217,data/fish/tu4images/tu4_77_01.png,tu4_77_01.png,tu4images,TU,4,77,1,1.0,128,unlabeled
16218,data/fish/tu4images/tu4_5_09.png,tu4_5_09.png,tu4images,TU,4,5,9,0.0,85,unlabeled
16219,data/fish/tu4images/tu4_5_03.png,tu4_5_03.png,tu4images,TU,4,5,3,4.0,85,unlabeled
16220,data/fish/tu4images/tu4_48_05.png,tu4_48_05.png,tu4images,TU,4,48,5,6.0,156,unlabeled


In [67]:
# combined_final.to_csv('data/fish/simCLR_endtoend/final_results.csv', index = False)

## RL

In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from tqdm import tqdm

# -----------------------------
# 1️⃣ PPO Environment 정의
# -----------------------------
class FishClassificationEnv:
    """
    단일 이미지 분류 환경 (Single-episode classification environment)
    각 step에서:
      - state: image tensor
      - action: predicted class index
      - reward: 1 (correct) or 0 (incorrect)
    """
    def __init__(self, dataloader, device):
        self.dataloader = dataloader
        self.device = device
        self.iterator = iter(dataloader)
        self.current_batch = None
        self.index = 0

    def reset(self):
        try:
            self.current_batch = next(self.iterator)
        except StopIteration:
            self.iterator = iter(self.dataloader)
            self.current_batch = next(self.iterator)
        imgs, labels = self.current_batch
        self.index = 0
        return imgs[self.index].unsqueeze(0).to(self.device), labels[self.index].item()

    def step(self, action, label):
        reward = 1.0 if action == label else 0.0
        done = True  # single-episode
        return reward, done

    def next_state(self):
        """다음 이미지 가져오기"""
        self.index += 1
        if self.index >= len(self.current_batch[0]):
            self.reset()
        imgs, labels = self.current_batch
        return imgs[self.index].unsqueeze(0).to(self.device), labels[self.index].item()


# -----------------------------
# 2️⃣ PPO Agent 정의
# -----------------------------
class PPOAgent(nn.Module):
    """
    PPO Agent: SimCLR backbone (frozen) + Policy/Value head
    """
    def __init__(self, backbone, feature_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        for param in self.backbone.parameters():
            param.requires_grad = False  # freeze backbone

        # PPO policy/value heads
        self.policy_head = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        self.value_head = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        logits = self.policy_head(features)
        values = self.value_head(features)
        return logits, values


# -----------------------------
# 3️⃣ PPO Update 함수 정의
# -----------------------------
class PPOTrainer:
    def __init__(self, model, lr=3e-4, gamma=0.99, clip_eps=0.2, value_coef=0.5, entropy_coef=0.03):
        self.model = model
        self.optimizer = optim.Adam(
            list(model.policy_head.parameters()) + list(model.value_head.parameters()),
            lr=lr
        )
        self.gamma = gamma
        self.clip_eps = clip_eps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

    def compute_returns(self, rewards, gamma):
        R = 0
        returns = []
        for r in reversed(rewards):
            R = r + gamma * R
            returns.insert(0, R)
        return torch.tensor(returns, dtype=torch.float32)

    def update(self, states, actions, old_log_probs, returns, advantages):
        logits, values = self.model(states)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions)
        entropy = dist.entropy().mean()

        # PPO ratio
        ratio = torch.exp(new_log_probs - old_log_probs.detach())
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        critic_loss = F.mse_loss(values.squeeze(), returns)
        loss = actor_loss + self.value_coef * critic_loss - self.entropy_coef * entropy

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item(), actor_loss.item(), critic_loss.item()


# -----------------------------
# 4️⃣ PPO 학습 루프
# -----------------------------
def train_ppo(backbone, train_loader, num_classes, device, num_epochs=5):
    model = PPOAgent(backbone, feature_dim=512, num_classes=num_classes).to(device)
    trainer = PPOTrainer(model)

    env = FishClassificationEnv(train_loader, device)
    print("Start PPO training...")

    for epoch in range(num_epochs):
        all_rewards = []
        losses = []
        for batch in tqdm(train_loader):
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)
            batch_size = imgs.size(0)

            states = imgs
            logits, values = model(states)
            dist = Categorical(logits=logits)
            actions = dist.sample()
            log_probs = dist.log_prob(actions)

            # Reward 계산 (정답 맞추면 +1)
            rewards = (actions == labels).float()

            returns = trainer.compute_returns(rewards, trainer.gamma).to(device)
            advantages = returns - values.detach().squeeze()

            loss, a_loss, c_loss = trainer.update(
                states, actions, log_probs, returns, advantages
            )
            losses.append(loss)
            all_rewards.extend(rewards.cpu().numpy())

        avg_reward = sum(all_rewards) / len(all_rewards)
        avg_loss = sum(losses) / len(losses)
        print(f"[Epoch {epoch+1}/{num_epochs}] Avg Reward: {avg_reward:.4f}, Loss: {avg_loss:.4f}")

    print("✅ PPO training completed.")
    return model


# -----------------------------
# 5️⃣ 실행
# -----------------------------
ppo_model = train_ppo(backbone, train_loader, num_classes, device, num_epochs=10)
torch.save(ppo_model.state_dict(), "data/fish/simCLR_endtoend/ppo_policy_value.pth")

Start PPO training...


100%|██████████| 123/123 [00:33<00:00,  3.72it/s]


[Epoch 1/10] Avg Reward: 0.4357, Loss: 24.5271


100%|██████████| 123/123 [00:32<00:00,  3.83it/s]


[Epoch 2/10] Avg Reward: 0.7715, Loss: 60.2337


...

[Epoch 8/10] Avg Reward: 0.8399, Loss: 67.5048


100%|██████████| 123/123 [00:32<00:00,  3.84it/s]


[Epoch 9/10] Avg Reward: 0.8578, Loss: 70.9508


100%|██████████| 123/123 [00:31<00:00,  3.86it/s]

[Epoch 10/10] Avg Reward: 0.8670, Loss: 72.3838
✅ PPO training completed.


In [42]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

@torch.no_grad()
def evaluate_ppo_detailed(model, dataloader, device, class_names=None):
    """
    PPO 모델의 클래스별 정밀도, 재현율, F1-score 평가
    Detailed evaluation for PPO model
    ----------------------------------
    - Input : dataloader (test_loader)
    - Output: classification report + confusion matrix
    """
    model.eval()
    all_labels = []
    all_preds = []

    for imgs, labels in tqdm(dataloader):
        imgs, labels = imgs.to(device), labels.to(device)
        logits, _ = model(imgs)

        # 가장 확률이 높은 클래스 선택 (deterministic prediction)
        preds = torch.argmax(logits, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # sklearn classification report
    print("\n✅ Classification Report (per class):")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    # confusion matrix (optional)
    cm = confusion_matrix(all_labels, all_preds)
    print("\n🧩 Confusion Matrix:")
    print(cm)

    return all_labels, all_preds, cm


# -----------------------------
# 실행
# -----------------------------
# 클래스 이름 지정 (선택 사항)
class_names = ["0+", "1+", "2+", "3+", "4+", "5+", "Bad"]

# PPO 모델 로드
ppo_model = PPOAgent(backbone, feature_dim=512, num_classes=num_classes).to(device)
ppo_model.load_state_dict(torch.load("data/fish/simCLR_endtoend/ppo_policy_value.pth", map_location=device))

# 평가
labels, preds, cm = evaluate_ppo_detailed(ppo_model, test_loader, device, class_names)

100%|██████████| 34/34 [00:09<00:00,  3.53it/s]


✅ Classification Report (per class):
              precision    recall  f1-score   support

          0+     0.8197    0.9216    0.8677       370
          1+     0.6920    0.8949    0.7805       723
          2+     0.0000    0.0000    0.0000       135
          3+     0.0000    0.0000    0.0000        69
          4+     0.0000    0.0000    0.0000         8
          5+     0.0000    0.0000    0.0000        10
         Bad     0.9752    0.9315    0.9529       803

    accuracy                         0.8196      2118
   macro avg     0.3553    0.3926    0.3716      2118
weighted avg     0.7492    0.8196    0.7793      2118


🧩 Confusion Matrix:
[[341  27   0   0   0   0   2]
 [ 62 647   0   0   0   0  14]
 [  1 132   0   0   0   0   2]
 [  5  64   0   0   0   0   0]
 [  0   7   0   0   0   0   1]
 [  0  10   0   0   0   0   0]
 [  7  48   0   0   0   0 748]]



/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/jlc3q/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
